# Linear model → MoE

Standalone small-model assignment. See README.md for architecture and SETUP.md for installation. The stored outputs below inspect the actual bundled evidence; fresh training is an explicit optional step.


In [1]:
from pathlib import Path
import json
ROOT = Path.cwd()
assert (ROOT / 'run.py').is_file(), 'Start the notebook from Small_Linear_MoE'
r = json.loads((ROOT / 'results/linear_results.json').read_text())
print('Initial loss:', r['dense_pretraining'][0]['loss'])
print('Trained linear:', r['dense_pretraining'][-1]['loss'])
print('Continued MoE:', r['moe_continuation'][-1]['loss'])
print('Continued dense:', r['dense_continuation'][-1]['loss'])
print('Conversion max logit difference:', r['conversion_max_logit_difference'])


Initial loss: 5.541113742894137
Trained linear: 2.393201353841788
Continued MoE: 2.2335704091984003
Continued dense: 2.3524556857424934
Conversion max logit difference: 9.5367431640625e-07


![Learning curves](figures/linear_learning.png)

Loss is cross-entropy on 79,116 held-out byte targets, not Transformer token loss.


In [2]:
d = json.loads((ROOT / 'deep_dive/diagnostics.json').read_text())
print('Intervention cases:', len(d['experiments']))
for row in d['experiments']:
    print(row['policy'], row.get('expert', ''), 'k=', row.get('k', 2), 'loss=', round(row['loss'], 6))


Intervention cases: 22
normal  k= 2 loss= 2.23357
uniform  k= 2 loss= 2.277916
fixed  k= 2 loss= 2.371851
all_off  k= 2 loss= 5.545177
normal  k= 1 loss= 2.243434
normal  k= 3 loss= 2.247976
normal  k= 4 loss= 2.2572
zero 0 k= 2 loss= 2.612155
zero 1 k= 2 loss= 2.680011
zero 2 k= 2 loss= 2.632155
zero 3 k= 2 loss= 2.710122
reroute 0 k= 2 loss= 2.281066
reroute 1 k= 2 loss= 2.308489
reroute 2 k= 2 loss= 2.287967
reroute 3 k= 2 loss= 2.285957
single 0 k= 2 loss= 2.460092
single 1 k= 2 loss= 2.474089
single 2 k= 2 loss= 2.497146
single 3 k= 2 loss= 2.52062
random  k= 2 loss= 2.408167
random  k= 2 loss= 2.40901
random  k= 2 loss= 2.409986


![Routing and shutdown](deep_dive/routing_and_shutdown.png)

Removing output changes logit scale as well as information. Rerouting tests substitution, not permanent pruning plus retraining.


In [3]:
v = json.loads((ROOT / 'validation/reproduction/REPRODUCTION.json').read_text())
print('Fresh independent-copy reproduction:', v['status'])
print('Deep-dive parity:', v['deep_dive'])
print('Environment:', v['evaluation']['environment'])


Fresh independent-copy reproduction: PASS
Deep-dive parity: {'status': 'PASS', 'intervention_cases': 22, 'balancing_arms': 6, 'max_absolute_reference_loss_difference': 0.0, 'tolerance': 0.0001}
Environment: {'python': '3.12.5', 'torch': '2.5.1+cu121', 'numpy': '1.26.4', 'cuda': '12.1', 'gpu': 'NVIDIA GeForce RTX 3070 Laptop GPU', 'platform': 'Windows-11-10.0.26300-SP0'}


## Two-seed results and measured cost


In [4]:
study = json.loads((ROOT/'results/extended_study/results.json').read_text())['stages']
for row in study:
    print(row['seed'], row['arm'], 'loss:', round(row['history'][-1]['loss'],6), 'targets/s:', round(row['training_targets_per_second']), 'peak MiB:', round(row['peak_training_allocated_bytes']/2**20,2))


31415 pretrained loss: 2.393201 targets/s: 33338 peak MiB: 36.24
31415 dense_control loss: 2.352456 targets/s: 31815 peak MiB: 36.24
31415 moe loss: 2.23357 targets/s: 30285 peak MiB: 48.67
27182 pretrained loss: 2.391488 targets/s: 34086 peak MiB: 36.24
27182 dense_control loss: 2.351383 targets/s: 33408 peak MiB: 36.24
27182 moe loss: 2.233629 targets/s: 29916 peak MiB: 48.75


![Two-seed learning](figures/two_seed_learning.png)

![Measured cost](figures/training_cost.png)


In [5]:
mix = json.loads((ROOT/'data/source_mix.json').read_text())
print('Source attribution:', mix['status'])
print(json.dumps(mix['counts_by_source_group'], indent=2))


Source attribution: PASS
{
  "train": {
    "long_context": 99919,
    "general": 102993,
    "indic": 191247,
    "agentic": 16170,
    "code": 180882,
    "science_math": 145270,
    "cosmopedia_v2": 42503,
    "wikipedia_en": 47772,
    "fineweb_edu": 170255,
    "reasoning": 2989
  },
  "val": {
    "agentic": 4151,
    "code": 7995,
    "general": 8989,
    "indic": 10327,
    "long_context": 9235,
    "reasoning": 1986,
    "science_math": 9774,
    "fineweb_edu": 8395,
    "wikipedia_en": 8108,
    "cosmopedia_v2": 10156
  }
}


In [6]:
import subprocess, sys
RUN_FRESH_TRAINING = False
if RUN_FRESH_TRAINING:
    subprocess.run([sys.executable, str(ROOT/'run.py'), 'reproduce', '--output', str(ROOT/'runs/notebook_reproduction')], check=True)
else:
    print('Training is off. Set RUN_FRESH_TRAINING=True on a CUDA machine to run the full experiment into a new output folder.')


Training is off. Set RUN_FRESH_TRAINING=True on a CUDA machine to run the full experiment into a new output folder.
